In [1]:
import pandas as pd
import json
import yaml
import seaborn as sns
from pathlib import Path

In [2]:
folder = Path('..', 'outputs', '002.experiment')

experiments = [ p for p in folder.iterdir()
                if p.is_dir() and Path(p, 'eval_results.json').exists() ]

for p in experiments: print(p.name)

d324ce601ed7140ae28bc58c3c344c7a
2e2d6e67655991b292444f54dd54eda3
ac1d2055b8c9d1f0885551a205f9a4ca
a4ebdb56de104f943b49d455b4d50a2b
770e0acb28df81599b31f15fd904fea2


In [3]:
def get_params(dir):
    params_file = Path(dir, 'parameters.yaml')
    parameters = yaml.safe_load(params_file.read_text())
    # parameters['folder'] = dir.name
    return parameters

def get_eval_results(dir):
    eval_file = Path(dir, 'eval_results.json')
    eval_results = json.loads(eval_file.read_text())
    eval_results['folder'] = dir.name
    return eval_results

df_params = pd.DataFrame([ get_params(p) for p in experiments ])
df_results = pd.DataFrame([ get_eval_results(p) for p in experiments])

df = pd.merge(df_params, df_results, left_on='_hash_id', right_on='folder')

teacher_keys = df['teachers_keys'].loc[0] # ['t5', 'llama']

df = pd.concat([
    df,
    df['teachers_weights'].apply(pd.Series, index=teacher_keys)
], axis=1)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 35 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   _hash_id                 5 non-null      object 
 1   _timestamp               5 non-null      object 
 2   batch_size               5 non-null      int64  
 3   bf16                     5 non-null      bool   
 4   dataset                  5 non-null      object 
 5   eval_steps               5 non-null      int64  
 6   experiment               5 non-null      object 
 7   from_pretrained          5 non-null      object 
 8   generation_max_length    5 non-null      int64  
 9   grad_steps               5 non-null      int64  
 10  local_rank               5 non-null      int64  
 11  logging_strategy         5 non-null      object 
 12  lr                       5 non-null      float64
 13  max_input_length         5 non-null      int64  
 14  max_steps                5 non

In [5]:
df[
    ['dataset', '_timestamp', 'lr', 'eval_accuracy', 'eval_token_accuracy', 'eval_loss']
].sort_values(by='eval_accuracy', ascending=False)

,dataset,_timestamp,lr,eval_accuracy,eval_token_accuracy,eval_loss
1,obqa,2025-11-12T21:35:13.418278,0.00050,0.716,0.91760,0.021173
2,obqa,2025-11-13T22:02:36.479126,0.00030,0.704,0.91772,0.017604
4,obqa,2025-11-14T08:15:18.125065,0.00080,0.696,0.91790,0.025356
0,obqa,2025-11-15T00:27:48.324493,0.00005,0.648,0.91754,0.012451
3,obqa,2025-11-13T14:23:13.375420,0.00500,0.000,0.00032,0.289217
